In [1]:
import gym
import talib
import numpy as np
import pandas as pd
import math
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import VecNormalize
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.utils import get_schedule_fn
from gym import spaces
import torch.nn as nn
import torch as th
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

sequence_len = 54

%run "../helpers/data_manipulator.py"

class TradingEnv(gym.Env):
    def __init__(self, data, initial_balance=10000):
        super(TradingEnv, self).__init__()
        self.data = data
        self.initial_balance = initial_balance
        self.current_step = 201  # Start at 201 for history
        self.cash_balance = self.initial_balance
        self.position = 0
        self.total_reward = 0
        self.last_price = self.data[self.current_step][0]
        self.average_buy_price = 0
        
        num_features = data.shape[1] - 1  # Exclude price column
        
        # Action space: 0 = Hold, 1 = Buy, 2 = Sell
        self.action_space = spaces.Box(low=0.0, high=1.0, shape=(1,), dtype=np.float32)
        
        # Observation space: Last sequence_len days of features
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(sequence_len, num_features), dtype=np.float32)
    
    def reset(self):
        self.current_step = 201
        self.cash_balance = self.initial_balance
        self.position = 0
        self.total_reward = 0
        self.last_price = self.data[self.current_step][0]
        self.average_buy_price = 0
        return self._next_observation()

    def _next_observation(self):
        return self.data[self.current_step-sequence_len:self.current_step, 1:]
    
    def step(self, action):
        current_price = self.data[self.current_step][0]
        next_price = self.data[self.current_step + 1][0] if self.current_step + 1 < len(self.data) - 1 else current_price
        reward = 0
        
        self.last_price = current_price
        today = current_price * self.position + self.cash_balance
        hypothetical_tomorrow = next_price * self.position + self.cash_balance + (self.cash_balance * 0.04 / 365)
        
        current_ratio = (current_price * self.position) / today
        
        if action > current_ratio and action > 0:
            invest_to_be = action * today
            current_invest = current_price * self.position
            diff = invest_to_be - current_invest
            amount = math.floor((diff / current_price) * 1.002)
            if amount > 0:
                if self.position > 0:
                    self.average_buy_price = ((self.average_buy_price * (self.position)) + current_price) / (self.position +1)
                else:
                    self.average_buy_price = current_price
                self.position += amount
                self.cash_balance -= (current_price * amount)
                self.cash_balance -= (current_price * amount) * 0.002
        elif action < current_ratio:
            invest_to_be = action * today
            current_invest = current_price * self.position
            diff = current_invest - invest_to_be
            count = math.ceil(diff / current_price)
            if count > 0:
                self.position -= count
                self.cash_balance += (current_price * count) * 0.998
        
        # self.cash_balance += (self.cash_balance * 0.04 / 365)  # Interest payment
        next_day = next_price * self.position + self.cash_balance
        reward += next_day / hypothetical_tomorrow - 1.0
        
        self.current_step += 1
        done = self.current_step >= len(self.data) - 1
        
        obs = self._next_observation()
        self.total_reward += reward
        return obs, reward, done, {}

class CNNFeatureExtractor(BaseFeaturesExtractor):
    def __init__(self, observation_space, features_dim=256):
        super(CNNFeatureExtractor, self).__init__(observation_space, features_dim)
        num_features = observation_space.shape[1]
        
        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels=num_features, out_channels=64, kernel_size=3, stride=2),
            nn.ReLU(),
            nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, stride=2),
            nn.ReLU(),
            nn.Conv1d(in_channels=128, out_channels=256, kernel_size=3, stride=2),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(256 * 5, features_dim),
            nn.ReLU()
        )
    
    def forward(self, observations):
        observations = observations.permute(0, 2, 1)  # Swap to (batch, features, time)
        return self.cnn(observations)

# Load and preprocess data
data = yf.download("UBER", start="2020-01-01", end="2024-10-01")
data = data.stack(level=1).reset_index()
data.reset_index()
data.columns.name = None
data = make_new_features_for(data)
normalized_data = normalize_data(data, "UBER")

normalized_data["Close"] = data["Close"]
data = normalized_data
# data = data[["Close"] + lstm_features].to_numpy()
data = data[["Close"] + [
    "Close-Prev-Close"
    ,"200D-Ratio"
    ,"MACD_SIG", "MFI3", "RSI"
                         , "RSI3", "POCR", "fastk", "fastd", "SlowK1", "willr3", "willr2"
                         , "willr1"
]]

data['MFI3'] = data['MFI3'].apply(lambda x: 0 if x < -1e-10 else x)

print("Maximum values:\n", data.iloc[200:].max())
print("\nMinimum values:\n", data.iloc[200:].min())


data = data.to_numpy()

print(data)

# Setup Environment
env = DummyVecEnv([lambda: TradingEnv(data)])
# env = VecNormalize(env, norm_obs=False, norm_reward=True)

# policy_kwargs = dict(
#     features_extractor_class=CNNFeatureExtractor,
#     features_extractor_kwargs=dict(features_dim=256)
# )

policy_kwargs = dict(activation_fn=torch.nn.Sigmoid, net_arch=[64, 32, 16])
# policy_kwargs = dict(activation_fn=torch.nn.Sigmoid, net_arch=[1024, 512, 256, 128], log_std_init=0.0)


model = PPO(
    "MlpPolicy", 
    env, 
    verbose=1, 
    gamma=0.99,
    # batch_size=256,
    learning_rate=0.0001,
    ent_coef=0.005,
    clip_range=0.2,
    vf_coef=1.0,
    gae_lambda=0.95,
    policy_kwargs=policy_kwargs
    # ,device=device  # Use GPU if available
)

model.learn(total_timesteps=1000000)

# Test Model
done = False
obs = env.reset()
# last_known_explianed_variance = None    
# cons_counter = 0

while not done:
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, done, _ = env.step(action)
    print(f"Step: {env.envs[0].current_step}, Action: {action}, Reward: {reward}, Position: {env.envs[0].position} Total Balance: {env.envs[0].cash_balance + env.envs[0].position * env.envs[0].last_price} Total Reward: {env.envs[0].total_reward}, Cash Balance: {env.envs[0].cash_balance}, Position Balance: {env.envs[0].position * env.envs[0].last_price}")

    # explained_variance = model.logger.name_to_value.get("train/explained_variance", None)
    # entropy_loss = model.logger.name_to_value.get("train/entropy_loss", None)

    # if entropy_loss > 0:
    #     print(f"Early stopping triggered! Entropy loss went above 0 {entropy_loss}.")
    
    # if explained_variance is not None:
    #     if last_known_explianed_variance is not None:
    #         if explianed_variance < last_known_explianed_variance:
    #             cons_counter += 1
    #             last_known_explianed_variance = explianed_variance
    #         else:
    #             cons_counter = 0
    #             last_known_explianed_variance = None
            
    # if last_known_explianed_variance is None and explained_variance > 0.9:
    #     last_known_explianed_variance = explained_variance

    # if cons_counter == 5:
    #     print(f"Early stopping triggered! Explained variance reached {explained_variance}.")
    #     break  # Stop the loop



NameError: name 'yf' is not defined

In [ ]:
print(f"\n\n\n Training Done \n\n\n")

In [ ]:
# Download new UBER data for prediction (example: past 30 days)
# new_data = yf.download("UBER", start="2020-01-01", end="2025-01-01")
data = yf.download("UBER", start="2024-01-01", end="2025-02-07")
data = data.stack(level=1).reset_index()
data.reset_index()
data.columns.name = None
data = make_new_features_for(data)
normalized_data = normalize_data(data, "UBER")

normalized_data["Close"] = data["Close"]
data = normalized_data
# data = data[["Close"] + lstm_features].to_numpy()
data = data[["Close"] + [
    "Close-Prev-Close"
    ,"200D-Ratio"
    ,"MACD_SIG", "MFI3", "RSI"
                         , "RSI3", "POCR", "fastk", "fastd", "SlowK1", "willr3", "willr2"
                         , "willr1"
]]

data['MFI3'] = data['MFI3'].apply(lambda x: 0 if x < -1e-10 else x)

print("Maximum values:\n", data.iloc[200:].max())
print("\nMinimum values:\n", data.iloc[200:].min())


data = data.to_numpy()

print(data)
# Create environment for new data
new_env = DummyVecEnv([lambda: TradingEnv(data)])

# Reset environment for predictions
obs = new_env.reset()
done = False

print("\n### PREDICTIONS ON NEW DATA ###\n")

while not done:
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, done, _ = new_env.step(action)
    print(f"Step: {new_env.envs[0].current_step}, Action: {action}, Reward: {reward}, Position: {new_env.envs[0].position} Last Price: {new_env.envs[0].last_price} Total Balance: {new_env.envs[0].cash_balance + new_env.envs[0].position * new_env.envs[0].last_price} Total Reward: {new_env.envs[0].total_reward}, Cash Balance: {new_env.envs[0].cash_balance}, Position Balance: {new_env.envs[0].position * new_env.envs[0].last_price}")
